# 07 Turkish Names MLP

Türkçe isimler veri kümesi ile MLP modelinin eğitimi, dev loss değerlendirmesi ve Bigram modeliyle karşılaştırmalı örnekleme.


In [ ]:
import csv
import io
import unicodedata
from urllib.request import urlopen
import random
import torch
import torch.nn.functional as F

url = "https://raw.githubusercontent.com/niyazikemer/turkce_isimler/main/turkce_isim.csv"
text = urlopen(url).read().decode("utf-8-sig")
rows = csv.DictReader(io.StringIO(text))
words = sorted(set(unicodedata.normalize("NFC", row["name"].strip().lower()) for row in rows))
words = [w for w in words if w and w.isalpha()]

chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)
print("Turkce alfabe boyutu (vocab_size):", vocab_size)
print("Toplam Turkce isim:", len(words))

block_size = 3
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

# Turkce MLP Modeli (Kaiming init + BatchNorm)
n_emb = 10
n_hidden = 200
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_emb), generator=g, requires_grad=True)
W1 = torch.randn((n_emb * block_size, n_hidden), generator=g) * (5/3) / ((n_emb * block_size)**0.5)
W1.requires_grad = True
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
W2.requires_grad = True
b2 = torch.randn(vocab_size, generator=g) * 0
b2.requires_grad = True

bngain = torch.ones((1, n_hidden), requires_grad=True)
bnbias = torch.zeros((1, n_hidden), requires_grad=True)
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]

for i in range(25000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C[Xtr[ix]].view(-1, n_emb * block_size)
    hpreact = emb @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)
    hpreact_norm = (hpreact - bnmean) / (bnstd + 1e-5) * bngain + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmean
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstd
    h = torch.tanh(hpreact_norm)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 18000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

# Dev Loss
emb_dev = C[Xdev].view(-1, n_emb * block_size)
hpreact_dev = emb_dev @ W1
hpreact_dev_norm = (hpreact_dev - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
h_dev = torch.tanh(hpreact_dev_norm)
logits_dev = h_dev @ W2 + b2
dev_loss = F.cross_entropy(logits_dev, Ydev)
print("Turkce MLP Dev Loss:", dev_loss.item())

# Karsilastirma icin Bigram Modeli
N_bigram = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)
for w in words[:n1]:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(chs, chs[1:]):
        N_bigram[stoi[ch1], stoi[ch2]] += 1
P_bigram = (N_bigram + 1).float()
P_bigram /= P_bigram.sum(1, keepdim=True)

# Yan Yana Ornekleme (Bigram vs MLP)
g_sample = torch.Generator().manual_seed(42)
print("\n" + "="*50)
print(f"{'No':<4} {'Bigram Uretimi':<20} {'MLP Uretimi':<20}")
print("="*50)

for idx in range(10):
    # Bigram
    out_bi = []
    ix_bi = 0
    while True:
        ix_bi = torch.multinomial(P_bigram[ix_bi], 1, replacement=True, generator=g_sample).item()
        if ix_bi == 0:
            break
        out_bi.append(itos[ix_bi])
        
    # MLP
    out_mlp = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])].view(1, -1)
        hpreact = emb @ W1
        hpreact_norm = (hpreact - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
        h = torch.tanh(hpreact_norm)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix_mlp = torch.multinomial(probs, num_samples=1, generator=g_sample).item()
        context = context[1:] + [ix_mlp]
        if ix_mlp == 0:
            break
        out_mlp.append(itos[ix_mlp])
        
    print(f"{idx+1:<4} {''.join(out_bi):<20} {''.join(out_mlp):<20}")
